# 8. Dataclasses

Dataclasses are Python's answer to Java's boilerplate problem. They auto-generate `__init__`, `__repr__`, `__eq__`, and more — just from type-annotated fields. Think of them as **Python's version of Java Records or Lombok's `@Data`**.

This notebook covers: `@dataclass`, default values, `field()`, `frozen`, `__post_init__`, `slots`, ordering, `kw_only`, `asdict()`/`astuple()`, inheritance, and comparison with NamedTuple.

### 8.1 Basic Dataclass

**☕ JAVA (Lombok):**
```java
@Data
public class Employee {
    private String name;
    private String department;
    private double salary;
}
```

**☕ JAVA 16+ (Record):**
```java
public record Employee(String name, String department, double salary) {}
```

**🐍 PYTHON:** `@dataclass` auto-generates `__init__`, `__repr__`, and `__eq__` from the annotated fields.

In [2]:
from dataclasses import dataclass, field, asdict, astuple

@dataclass
class Employee: 
    # name, department, salary, active
    name: str
    department: str
    salary: float
    active: bool = True # default value

emp = Employee("Alice", "AI Engineer", 100_000)
print(emp)

emp2 = Employee("Alice", "AI Engineer", 100_000)
print(emp == emp2)

Employee(name='Alice', department='AI Engineer', salary=100000, active=True)
True


In [ ]:
new_emp = Employee("Alice", "Software developer", 52_000, "No" ) # We haven't validation for the values. 
print(new_emp.active)

No


### 8.2 The `field()` Function — Advanced Defaults

**☕ JAVA:** No equivalent — mutable defaults aren't a problem in Java constructors.

**🐍 PYTHON:** ⚠️ You **cannot** use a mutable default directly (like `tags: list = []`). Use `field(default_factory=list)` instead.

In [3]:
@dataclass
class Article:
    title: str
    author: str
    views: int = 0 #default value
    # tags: list[str] = [] # this give error 
    tags: list[str] = field(default_factory=list) # Mutable defaults
    _internal_id: str = field(repr=False, default="")

a1 = Article("Python Tips", "Bob")
a1.tags.append("python")

a2 = Article("Java Tips", "Helen")
a2.tags.append("java")

print(f"a1 tags: {a1.tags}")
print(f"a2 tags: {a2.tags}")
print(a1)
print(a2)

a1 tags: ['python']
a2 tags: ['java']
Article(title='Python Tips', author='Bob', views=0, tags=['python'])
Article(title='Java Tips', author='Helen', views=0, tags=['java'])


### 8.3 `__post_init__` — Custom Initialization Logic

**☕ JAVA:** Validation logic goes in the constructor or a `@PostConstruct` method.

**🐍 PYTHON:** `__post_init__` runs **after** the auto-generated `__init__` — perfect for validation or computed fields.

###  Γιατί χρησιμοποιούμε το `field()` στα dataclasses
 
Το `field()` από το Python `dataclasses` module χρησιμοποιείται όταν θέλουμε **να ελέγξουμε λεπτομερώς τη συμπεριφορά ενός πεδίου** σε ένα `@dataclass`.
 
1. Mutable defaults (ΠΟΛΥ σημαντικό): tags: list[str] = field(default_factory=list) # Δημιουργεί νέα λίστα για κάθε instance
2. Έλεγχος στο __repr__ : _internal_id: str = field(repr=False)  # Δεν εμφανίζεται στο print(obj)
3. Έλεγχος σε συγκρίσεις (__eq__): timestamp: float = field(compare=False) # Αγνοείται σε equality checks
4. Αφαίρεση από constructor (__init__): id: int = field(init=False, default=0) # Δεν περνάει ως parameter στο __init__
5. Metadata (advanced usage): name: str = field(metadata={"max_length": 50}) # Χρήσιμο για frameworks (validation, ORM, κτλ)


In [2]:
from dataclasses import dataclass, field

@dataclass
class User:
    name: str
    age: int
    is_adult: bool = field(init=False)

    def __new__(cls, *args, **kwds):
        print("1. __new__ called")
        instanse = super().__new__(cls)
        return instanse
    
    # def __init__(self, name: str, age: int):
    #     print("2.__init__ called")
    #     self.name = name
    #     self.age  = age
    #     self.__post_init__() # If we have @dataclass this is automatically added diferrent we need to add this if we have __post_init__ with custom __init__. 

    def __post_init__(self):
        print("3.__post_init__ called")
        self.is_adult = self.age >= 18


In [13]:
alice = User("Alice", 20)
print(alice)

1. __new__ called
3.__post_init__ called
User(name='Alice', age=20, is_adult=True)


In [20]:
@dataclass
class Rectangle:
    width: float
    height: float
    area: float = field(init=False)

    def __post_init__(self):
        if self.width <= 0 or self.height <= 0:
            raise ValueError("Dimensions can't be negatives")
        self.area = self.width * self.height

try:
    bad_rec = Rectangle(-1, 10)
    good_rec = Rectangle(2, 10)
    print(good_rec)
except ValueError as e:
    print(f"Error: {e}")
    

Error: Dimensions can't be negatives


### 8.4 `frozen=True` — Immutable Dataclasses

**☕ JAVA:** `record` types are immutable by default.

**🐍 PYTHON:** `@dataclass(frozen=True)` makes instances immutable — they can't be modified after creation. This also auto-generates `__hash__`, so they can be used in sets/dicts.

In [28]:
@dataclass(frozen=True)
class Point:
    x: float
    y:float

p = Point(10, 20)
print(p)

try:
    p.x = 100
    print(p)
except Exception as e:
    print(e)



Point(x=10, y=20)
cannot assign to field 'x'


### 8.5 `order=True` — Sortable Dataclasses

**☕ JAVA:** Implement `Comparable<T>` with `compareTo()`.

**🐍 PYTHON:** `@dataclass(order=True)` auto-generates `<`, `<=`, `>`, `>=` — comparing **field by field** in declaration order.

In [ ]:
@dataclass(order=True)
class Student:
    gpa: float
    name: str # If we don't want to compare this = field(compare=False)

students = [
    Student(3.5, "Charlie"),
    Student(3.9, "Alice"),
    Student(3.5, "Bob")
]

for st in sorted(students):
    print(f"{st.gpa}- {st.name}")

print(f"Highest GPA: {max(students)}")

3.5- Bob
3.5- Charlie
3.9- Alice
Highest GPA: Student(gpa=3.9, name='Alice')


### 8.6 `kw_only=True` — Keyword-Only Arguments (Python 3.10+)

**☕ JAVA:** Named parameters don't exist — you rely on IDE hints or the Builder pattern.

**🐍 PYTHON:** `kw_only=True` forces callers to use keyword arguments, preventing positional mistakes. You can also mark individual fields with `field(kw_only=True)`.

In [35]:
@dataclass(kw_only=True)
class DatabaseConfig:
    host: str
    port: int = 5432
    database: str = "mydb"
    ssl: bool = True

config = DatabaseConfig(host="localhost", database="prod") 
print(config)


DatabaseConfig(host='localhost', port=5432, database='prod', ssl=True)


In [36]:
try:
    bad_config = DatabaseConfig("localhost", 7210) 
except TypeError as e:
    print(f"Error: {e}")

Error: DatabaseConfig.__init__() takes 1 positional argument but 3 were given


### Only specific fields to be keyword arguments only

In [37]:
@dataclass
class APIRequest:
    url: str                                        # Positional (required)
    method: str = "GET"                             # Positional (optional)
    timeout: int = field(default=30, kw_only=True)  # Keyword only
    retries: int = field(default=3, kw_only=True)   # Keyword only

req = APIRequest("https://api.example.com", "POST", timeout=60)
print(req)

APIRequest(url='https://api.example.com', method='POST', timeout=60, retries=3)


### 8.7 `asdict()` and `astuple()` — Serialization Helpers

**☕ JAVA:** You'd write `toMap()` or use a library like Jackson.

**🐍 PYTHON:** Built-in `asdict()` and `astuple()` convert dataclasses to dicts/tuples — essential for JSON serialization, database insertion, or API responses.

In [38]:
import json
 
@dataclass
class Address:
    street: str
    city: str
    country: str = "Greece"
 
@dataclass
class Person:
    name: str
    age: int
    address: Address   # Nested dataclass!
 
person = Person("Alice", 30, Address("123 Main St", "Athens"))
 
# asdict — recursively converts to dict (including nested dataclasses!)
person_dict = asdict(person)
print(f"Dict: {person_dict}")
print(f"JSON: {json.dumps(person_dict, indent=2)}")
 
# astuple — converts to tuple (flat)
print(f"\nTuple: {astuple(person)}")

Dict: {'name': 'Alice', 'age': 30, 'address': {'street': '123 Main St', 'city': 'Athens', 'country': 'Greece'}}
JSON: {
  "name": "Alice",
  "age": 30,
  "address": {
    "street": "123 Main St",
    "city": "Athens",
    "country": "Greece"
  }
}

Tuple: ('Alice', 30, ('123 Main St', 'Athens', 'Greece'))


> 💡 **Key feature:** `asdict()` handles **nested dataclasses** recursively — the `Address` inside `Person` becomes a nested dict automatically.

### 8.8 Dataclass Inheritance

**☕ JAVA:** `record` types **cannot** be extended — they are implicitly `final`.

**🐍 PYTHON:** Dataclasses support full inheritance! Child fields are appended after parent fields in the constructor.

In [3]:
@dataclass
class Vehicle:
    make: str
    year: int
 
@dataclass
class Car(Vehicle):
    doors: int = 4
    fuel: str = "petrol"
 
@dataclass
class ElectricCar(Car):
    battery_kwh: float = 75.0
    fuel: str = "electric"   # Override parent default!
 
car = Car("Toyota", 2024)
ev = ElectricCar("Tesla", 2024, doors=4, battery_kwh=100)
 
print(f"Car: {car}")
print(f"EV:  {ev}")
print(f"\nIs Car: {isinstance(ev, Car)}")        # True
print(f"Is Vehicle: {isinstance(ev, Vehicle)}")  # True

Car: Car(make='Toyota', year=2024, doors=4, fuel='petrol')
EV:  ElectricCar(make='Tesla', year=2024, doors=4, fuel='electric', battery_kwh=100)

Is Car: True
Is Vehicle: True


> ⚠️ **Gotcha:** If a parent has fields with defaults, ALL child fields must also have defaults (same Python rule as regular function arguments — no positional after keyword).

### 8.9 `slots=True` — Memory Optimization (Python 3.10+)

**🐍 PYTHON:** `@dataclass(slots=True)` auto-generates `__slots__` — faster attribute access and lower memory usage. No dynamic attribute addition.

In [5]:
@dataclass(slots=True)
class Coordinate:
    x: float
    y: float
    z: float = 0.0

coord = Coordinate(1, 2, 3)
print(f"coord: {coord}")

try:
    coord.w = 4
except AttributeError as e:
    print(f"Error: {e}")

coord: Coordinate(x=1, y=2, z=3)
Error: 'Coordinate' object has no attribute 'w'


### 8.10 Dataclass vs NamedTuple

| Feature | `@dataclass` | `NamedTuple` |
|---------|-------------|-------------|
| Mutable by default | ✅ Yes | ❌ Always immutable |
| Can be frozen | ✅ `frozen=True` | Always frozen |
| Inheritance | ✅ Full class | Limited |
| Is a tuple? | ❌ No | ✅ Yes (supports indexing, unpacking) |
| `__hash__` | Only if frozen or explicit | ✅ Auto |
| Methods | ✅ Full support | ✅ Full support |
| Best for | Most data classes | When you need tuple behavior |

In [6]:
from typing import NamedTuple

class PointNT(NamedTuple):
    x: float
    y: float

p = PointNT(3, 4)
print(f"NamedTuple: {p}")

NamedTuple: PointNT(x=3, y=4)


In [8]:
x, y = p
print(x, ",", y)
print(f"Index: {p[0]}, {p[1]}")

3 , 4
Index: 3, 4


---

## 🧪 Try It Yourself

**Exercise 1:** Create a `@dataclass` `Product` with `name`, `price`, `quantity` (default 0). Add a computed `total_value` property via `__post_init__`. Make it sortable by price using `order=True`. Use `asdict()` to serialize it to JSON.

In [ ]:
# Exercise 1: Your code here


**Exercise 2:** Create a `frozen=True` `Color` dataclass with `r`, `g`, `b` (all `int`). Validate in `__post_init__` that values are 0–255. Demonstrate it works in a `set`.

In [ ]:
# Exercise 2: Your code here


**Exercise 3:** Create a dataclass hierarchy: `Shape` (base with `color: str`), `Circle(Shape)` with `radius`, and `Rectangle(Shape)` with `width` and `height`. Use `kw_only=True` on `Shape` and add computed `area` via `__post_init__`. Serialize to dict using `asdict()`.

In [ ]:
# Exercise 3: Your code here


---

## 📝 Key Takeaways: Java → Python

| Concept | Java | Python |
|---------|------|--------|
| Auto-generated class | Lombok `@Data` / `record` | `@dataclass` |
| Auto `__init__` | Lombok `@AllArgsConstructor` | ✅ Default |
| Auto `__repr__` | Lombok `@ToString` | ✅ Default |
| Auto `__eq__` | Lombok `@EqualsAndHashCode` | ✅ Default |
| Immutable | `record` (Java 16+) | `@dataclass(frozen=True)` |
| Sortable | `Comparable<T>` interface | `@dataclass(order=True)` |
| Keyword-only args | Builder pattern | `@dataclass(kw_only=True)` |
| Mutable default | Not a problem | `field(default_factory=...)` required! |
| Post-construction | `@PostConstruct` | `__post_init__` |
| To dict/tuple | Jackson / manual `toMap()` | `asdict()` / `astuple()` |
| Inheritance | `record` is `final` — can't extend | ✅ Full inheritance support |
| Memory optimization | N/A | `@dataclass(slots=True)` |
| Hidden from repr | Lombok `@ToString.Exclude` | `field(repr=False)` |